<a href="https://colab.research.google.com/github/andrewdigiacomo27/econ8310-assignment1/blob/main/Econ8310_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4
## Econ 8310 - Business Forecasting

This assignment will make use of the bayesian statistical models covered in Lessons 10 to 12.

A/B Testing is a critical concept in data science, and for many companies one of the most relevant applications of data-driven decision-making. In order to improve product offerings, marketing campaigns, user interfaces, and many other user-facing interactions, scientists and engineers create experiments to determine the efficacy of proposed changes. Users are then randomly assigned to either the treatment or control group, and their behavior is recorded.
If the changes that the treatment group is exposed to can be measured to have a benefit in the metric of interest, then those changes are scaled up and rolled out to across all interactions.
Below is a short video detailing the A/B Testing process, in case you want to learn a bit more:
[https://youtu.be/DUNk4GPZ9bw](https://youtu.be/DUNk4GPZ9bw)

For this assignment, you will use an A/B test data set, which was pulled from the Kaggle website (https://www.kaggle.com/datasets/yufengsui/mobile-games-ab-testing). I have added the data from the page into Codio for you. It can be found in the cookie_cats.csv file in the file tree. It can also be found at [https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv](https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv)

The variables are defined as follows:

| Variable Name  | Definition |
|----------------|----|
| userid         | A unique number that identifies each player  |
| version        | Whether the player was put in the control group (gate_30 - a gate at level 30) or the group with the moved gate (gate_40 - a gate at level 40) |
| sum_gamerounds | The number of game rounds played by the player during the first 14 days after install.  |
| retention1     | Did the player come back and play 1 day after installing?     |
| retention7     | Did the player come back and play 7 days after installing?    |               

### The questions

You will be asked to answer the following questions in a small quiz on Canvas:
1. What was the effect of moving the gate from level 30 to level 40 on 1-day retention rates?
2. What was the effect of moving the gate from level 30 to level 40 on 7-day retention rates?
3. What was the biggest challenge for you in completing this assignment?

You will also be asked to submit a URL to your forked GitHub repository containing your code used to answer these questions.

In [24]:
import pymc as pm
import pandas as pd
import numpy as np
import arviz as az


In [ ]:
data = pd.read_csv("https://github.com/dustywhite7/Econ8310/raw/master/AssignmentData/cookie_cats.csv")
print(data)

        userid  version  sum_gamerounds  retention_1  retention_7
0          116  gate_30               3        False        False
1          337  gate_30              38         True        False
2          377  gate_40             165         True        False
3          483  gate_40               1        False        False
4          488  gate_40             179         True         True
...        ...      ...             ...          ...          ...
90184  9999441  gate_40              97         True        False
90185  9999479  gate_40              30        False        False
90186  9999710  gate_30              28         True        False
90187  9999768  gate_40              51         True        False
90188  9999861  gate_40              16        False        False

[90189 rows x 5 columns]


In [19]:
data['retention_1'] = data['retention_1'].astype(int)
data['retention_7'] = data['retention_7'].astype(int)
gate30 = data[data['version'] == 'gate_30']
gate40 = data[data['version'] == 'gate_40']
print(gate30)
print(gate40)


In [25]:
# Retention 1

val30 = gate30['retention_1'].values
val40 = gate40['retention_1'].values

with pm.Model() as gate_retention_model:
  prior30 = pm.Beta('prior30', alpha=1, beta=1)
  prior40 = pm.Beta('prior40', alpha=1, beta=1)

  observed30 = pm.Bernoulli('observed30', p=prior30, observed=val30)
  observed40 = pm.Bernoulli('observed40', p=prior40, observed=val40)

  difference = pm.Deterministic('difference', prior40-prior30)

  trace1 = pm.sample()

az.summary(trace1, var_names=["prior30", "prior40", "difference"])


Output()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
prior30,0.448,0.002,0.444,0.453,0.0,0.0,1889.0,1281.0,1.0
prior40,0.442,0.002,0.438,0.446,0.0,0.0,1939.0,1311.0,1.0
difference,-0.006,0.003,-0.012,-0.000,0.0,0.0,1856.0,1407.0,1.0


In [26]:
# Retention 7

val30 = gate30['retention_7'].values
val40 = gate40['retention_7'].values

with pm.Model() as gate_retention_model:
  prior30 = pm.Beta('prior30', alpha=1, beta=1)
  prior40 = pm.Beta('prior40', alpha=1, beta=1)

  observed30 = pm.Bernoulli('observed30', p=prior30, observed=val30)
  observed40 = pm.Bernoulli('observed40', p=prior40, observed=val40)

  difference = pm.Deterministic('difference', prior40-prior30)

  trace7 = pm.sample()

az.summary(trace7, var_names=["prior30", "prior40", "difference"])

Output()

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
prior30,0.190,0.002,0.187,0.194,0.0,0.0,2000.0,1158.0,1.0
prior40,0.182,0.002,0.179,0.185,0.0,0.0,2130.0,1530.0,1.0
difference,-0.008,0.002,-0.013,-0.003,0.0,0.0,2124.0,1415.0,1.0
